In [1]:
import os
import json
import numpy as np
import random
import time
import re
import tiktoken
from google import genai
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from collections import defaultdict
from openai import OpenAI

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


In [2]:

# --- Configuration ---
EMBEDDING_DIM = 512  # Dimension for question and answer embeddings

UPDATE_FREQUENCY = 1    # Update JSON records after every question


# TODO: configure more parameters here
# Regularization parameter λ, exploration coefficient γ, step size η, network width m, 
# gradient descent steps J, LLM pool size M , batch size B_batch

LEARNING_RATE = 0.01 # 学习率
REG = 1 #正则化参数
M = 7 # LLM pool size
GEMMA = 1 # 探索系数
L = 3 # 2层隐藏层
BATCH_SIZE = 10 # B_batch
WIDTH = 100 # m
GD_STEPS = 10 # J

In [3]:
# --- CUDA / Device Setup ---
# 自动选择 GPU；若不可用则回退到 CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA runtime: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.get_device_name(torch.cuda.current_device())}")
else:
    print("Using CPU")


def to_device(x):
    """Move tensor/module to selected device."""
    if hasattr(x, "to"):
        return x.to(DEVICE)
    return x


# 可选：提高矩阵乘法性能（Ampere 及以上通常有效）
torch.set_float32_matmul_precision("high")

Torch version: 2.5.1+cu121
CUDA available: True
CUDA runtime: 12.1
GPU count: 1
Current GPU: NVIDIA GeForce RTX 2060


In [4]:
if DEVICE.type == "cuda":
    # 清理缓存（可选）
    torch.cuda.empty_cache()
    print("CUDA is ready for training/inference.")

CUDA is ready for training/inference.


In [5]:
from dotenv import load_dotenv, find_dotenv
from pathlib import Path

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    # Notebook 常在 Algorithms/ 目录运行，.env 在项目根目录
    candidates = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
    for p in candidates:
        if p.exists():
            dotenv_path = str(p)
            break
if dotenv_path:
    load_dotenv(dotenv_path, override=False)
    print("Loaded .env:", dotenv_path)
else:
    print("No .env found; please create one at project root.")


OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL")

Loaded .env: e:\LLM_DAG_Routing-main\.env


In [6]:
# --- Test two models via OpenRouter ---
Decompose_MODEL_NAME = "google/gemini-2.5-flash-lite-preview-09-2025"
GRADER_MODEL_NAME = "google/gemini-2.5-flash-lite-preview-09-2025"

assert OPENROUTER_API_KEY, "OPENROUTER_API_KEY 未设置，请检查 .env"
base_url = OPENROUTER_BASE_URL 

client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=base_url)

models_to_test = [Decompose_MODEL_NAME, GRADER_MODEL_NAME]
prompt = "请只回复：OK"

for model_name in models_to_test:
    try:
        resp = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=16,
            temperature=0.0,
        )
        text = resp.choices[0].message.content if resp.choices else "<no choice>"
        print(f"✅ {model_name} -> {text}")
    except Exception as e:
        print(f"❌ {model_name} failed: {type(e).__name__}: {e}")

✅ google/gemini-2.5-flash-lite-preview-09-2025 -> OK
✅ google/gemini-2.5-flash-lite-preview-09-2025 -> OK


In [7]:
# 从项目根目录的 model_config.py 加载模型列表
import sys
from pathlib import Path

# 当前 notebook 在 Algorithm/ 下，根目录是其上一级
project_root = Path.cwd() if (Path.cwd() / "model_config.py").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from model_config import MODELS_CONFIG

AVAILABLE_LLMS = list(MODELS_CONFIG.keys())
LLM_ID_DIM = len(AVAILABLE_LLMS)

print(f"Loaded {LLM_ID_DIM} models from model_config.py")
print(AVAILABLE_LLMS[:5])

Loaded 7 models from model_config.py
['qwen/qwen-2.5-7b-instruct', 'meta-llama/llama-3.1-8b-instruct', 'mistralai/mistral-small-3.2-24b-instruct', 'google/gemma-3-27b-it', 'meta-llama/llama-3.3-70b-instruct']


In [8]:
import json
import os

# ==========================================
# TODO: load train set data/fused_qa_500.json
# ==========================================
def load_dataset(file_path="Algorithms\dag_math_dag.json"):
    if not os.path.exists(file_path):
        print(f"⚠️ 警告: 数据集文件 {file_path} 不存在，请检查路径。")
        return {}
    with open(file_path, 'r', encoding='utf-8') as f:
        print(f"✅ 成功加载数据集: {file_path}")
        return json.load(f)

# ==========================================
# TODO: 创建记录模型回答的json文件
# ==========================================
class ResultRecorder:
    def __init__(self, output_file="execution_records.json"):
        self.output_file = output_file
        self.records = {}
        # 如果文件已存在，则追加记录
        if os.path.exists(output_file):
            with open(output_file, 'r', encoding='utf-8') as f:
                try:
                    self.records = json.load(f)
                except json.JSONDecodeError:
                    self.records = {}

    def add_record(self, problem_id: str, question: str, expected_answers: str, 
                   final_status: str, attempts: list):
        """
        按照规定的字典格式写入单条测试记录。
        attempts 应该是一个包含 step, task_desc, chosen_model, reward, output, llm_input_messages 的列表字典。
        """
        self.records[problem_id] = {
            "question": question,
            "answers": expected_answers,
            "final_status": final_status,
            "steps_taken": len(attempts),
            "attempts": attempts
        }
        self.save()

    def save(self):
        """将记录持久化到 JSON 文件中，支持 UPDATE_FREQUENCY"""
        with open(self.output_file, 'w', encoding='utf-8') as f:
            json.dump(self.records, f, ensure_ascii=False, indent=2)

In [9]:
# ==========================================
# 文本向量化模块 (Feature Extractor)
# ==========================================
EMBEDDING_MODEL = "BAAI/bge-small-zh-v1.5" # 使用支持中英双语的轻量级模型
EMBEDDING_DIM = 512

class FeatureExtractor:
    def __init__(self, model_name=EMBEDDING_MODEL, device=DEVICE):
        print(f"Loading embedding model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.model.to(device)
        print("Embedding model loaded successfully.")

    def get_feature_vector(self, text_context):
        # 如果输入为空，返回全0向量（防止报错）
        if not text_context or text_context.strip() == "":
            return np.zeros(EMBEDDING_DIM, dtype=np.float32)
            
        vector = self.model.encode(
            text_context, 
            convert_to_numpy=True, 
            normalize_embeddings=True
        )
        return vector.astype(np.float32)

# 实例化特征提取器
extractor = FeatureExtractor()

Loading embedding model: BAAI/bge-small-zh-v1.5...
Embedding model loaded successfully.


In [10]:
# --- 网络结构定义 ---
class LLMNet(nn.Module):
    def __init__(self, input_dim=512, width=100, L=3):
        super().__init__()
        num_hidden = max(L - 1, 1)
        layers = []
        in_dim = input_dim
        for _ in range(num_hidden):
            layers.append(nn.Linear(in_dim, width))
            layers.append(nn.ReLU())
            in_dim = width
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        if not torch.is_tensor(x):
            x = torch.tensor(x, dtype=torch.float32).to(DEVICE)
        if x.dim() == 1:
            x = x.unsqueeze(0)
        return self.net(x).squeeze(-1)

In [11]:
class DAGNode:
    def __init__(self, node_id, task_description, expected_output=""):
        self.node_id = node_id
        self.task_description = task_description
        self.expected_output = expected_output # 新增：存储该节点的标准答案
        self.predecessors = []
        self.successors = []
        self.layer = 0
        
        # 运行时状态
        self.input_context = ""
        self.output_content = ""
        self.selected_model = None
        self.feature_vector = None 
        self.reward = 0.0          

    def add_successor(self, node):
        if node not in self.successors:
            self.successors.append(node)
        if self not in node.predecessors:
            node.predecessors.append(self)

class DAGGraph:
    def __init__(self):
        self.problem_id = ""
        self.problem_description = ""
        self.nodes = {}

    def add_node(self, node_id, description, expected_output=""):
        if node_id not in self.nodes:
            self.nodes[node_id] = DAGNode(node_id, description, expected_output)
        return self.nodes[node_id]

    def add_edge(self, source_id, target_id):
        if source_id in self.nodes and target_id in self.nodes:
            self.nodes[source_id].add_successor(self.nodes[target_id])

    def compute_layers(self):
        #层级计算：Layer = max(predecessor_layers) + 1
        for node in self.nodes.values(): node.layer = 0
        for _ in range(len(self.nodes) + 1):
            for node in self.nodes.values():
                for pred in node.predecessors:
                    if node.layer < pred.layer + 1:
                        node.layer = pred.layer + 1
        layers = {}
        for node in self.nodes.values():
            if node.layer not in layers: layers[node.layer] = []
            layers[node.layer].append(node)
        return layers

In [12]:
def build_dag_from_ground_truth(problem_dict: dict) -> DAGGraph:
    """
    直接从预标注的 JSON 数据字典中构建 DAG。
    跳过 LLM 动态分解。
    """
    graph = DAGGraph()
    graph.problem_id = problem_dict.get("problem_id", "unknown_id")
    graph.problem_description = problem_dict.get("problem_description", "")
    
    nodes_data = problem_dict.get("nodes", [])
    
    # 1. 优先创建所有节点，并写入期望输出
    for n_data in nodes_data:
        graph.add_node(
            node_id=n_data["node_id"], 
            description=n_data["sub_task_description"],
            expected_output=n_data.get("expected_output", "")
        )
        
    # 2. 建立依赖边
    for n_data in nodes_data:
        target_id = n_data["node_id"]
        for source_id in n_data.get("dependency_node_id", []):
            graph.add_edge(source_id, target_id)
            
    print(f"✅ Successfully loaded Ground Truth DAG with {len(nodes_data)} nodes for Problem {graph.problem_id}.")
    return graph

In [13]:
import re

def execute_and_evaluate_dag(graph: DAGGraph, ucb_model, feature_extractor, client):
    """
    执行 DAG 并应用条件打分逻辑，同时记录 UCB 分数细节和 Token 消耗。
    """
    problem_desc = graph.problem_description
    layers_dict = graph.compute_layers()
    sorted_layers = sorted(layers_dict.keys())
    
    observations = [] 
    execution_attempts = [] 
    
    # 记录 Grader 裁判模型的额外开销
    total_grader_prompt_tokens = 0
    total_grader_completion_tokens = 0
    
    # ==========================================
    # 第一阶段：按拓扑序执行所有节点 (生成推理轨迹)
    # ==========================================
    for layer in sorted_layers:
        for node in layers_dict[layer]:
            parent_outputs = "\n".join([f"Result of prerequisite step {p.node_id}: {p.output_content}" for p in node.predecessors])
            context_text = f"Global background and problem: {problem_desc}\n{parent_outputs}\nCurrent derivation task: {node.task_description}"
            node.input_context = context_text
            
            x_tn = feature_extractor.get_feature_vector(context_text)
            node.feature_vector = x_tn
            
            # 获取最佳模型以及所有模型的 UCB 打分明细
            best_model, scores_info = ucb_model.select_model(x_tn)
            node.selected_model = best_model
            
            # 严格英文 Prompt，限制只输出答案
            prompt = f"Given the following background and the results of previous steps, complete the current derivation task. Output ONLY the final answer. Do not show your reasoning process, explanation, or any conversational filler.\n\n{context_text}"
            
            p_tokens = 0
            c_tokens = 0
            
            try:
                resp = client.chat.completions.create(
                    model=best_model,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=256,
                    temperature=0.1, 
                )
                node.output_content = resp.choices[0].message.content.strip()
                
                # 提取 Token 消耗（花费）
                if hasattr(resp, 'usage') and resp.usage:
                    p_tokens = resp.usage.prompt_tokens
                    c_tokens = resp.usage.completion_tokens
                    
            except Exception as e:
                print(f"节点 {node.node_id} 调用模型 {best_model} 失败: {e}")
                node.output_content = "Execution failed"
                
            # 记录执行尝试，加入 UCB 明细和 Token 消耗
            execution_attempts.append({
                "node_id": node.node_id,
                "task_desc": node.task_description,
                "chosen_model": node.selected_model,
                "ucb_scores_detail": scores_info,      # 记录选型打分细节
                "prompt_tokens": p_tokens,             # 记录输入花费
                "completion_tokens": c_tokens,         # 记录输出花费
                "output": node.output_content,
                "expected_output": node.expected_output,
                "reward": 0.0 
            })
            
    # ==========================================
    # 第二阶段：评估终端节点 (计算最终全局得分)
    # ==========================================
    terminal_nodes = [n for n in graph.nodes.values() if not n.successors]
    final_output = "\n".join([f"Final Conclusion {n.node_id}: {n.output_content}" for n in terminal_nodes])
    expected_final_output = "\n".join([f"Expected Conclusion {n.node_id}: {n.expected_output}" for n in terminal_nodes])
    
    grade_prompt_final = f"""
    Original Problem: {problem_desc}
    
    [Expected Final Conclusion]: 
    {expected_final_output}
    
    [Model's Actual Final Conclusion]: 
    {final_output}
    
    As a strict judge, evaluate whether the model's actual conclusion matches the meaning of the expected conclusion and correctly solves the original problem.
    - Output 1 if completely correct.
    - Output 0 if completely incorrect.
    - Output a float between 0.1 and 0.9 for partial correctness.
    Output ONLY the number, with no other characters.
    """
    
    try:
        resp_grade = client.chat.completions.create(
            model=GRADER_MODEL_NAME,
            messages=[{"role": "user", "content": grade_prompt_final}],
            max_tokens=10,
            temperature=0.0,
        )
        score_str = resp_grade.choices[0].message.content.strip()
        match = re.search(r"([0-9]*\.?[0-9]+)", score_str)
        final_score = float(match.group(1)) if match else 0.0
        
        # 累计裁判模型花费
        if hasattr(resp_grade, 'usage') and resp_grade.usage:
            total_grader_prompt_tokens += resp_grade.usage.prompt_tokens
            total_grader_completion_tokens += resp_grade.usage.completion_tokens
            
    except Exception as e:
        print(f"全局打分失败: {e}")
        final_score = 0.0
        
    final_score = max(0.0, min(1.0, final_score))
    
    # ==========================================
    # 第三阶段：条件奖励分配
    # ==========================================
    if final_score == 1.0:
        for node in graph.nodes.values():
            node.reward = 1.0
            observations.append((node.feature_vector, node.selected_model, node.reward))
            for attempt in execution_attempts:
                if attempt["node_id"] == node.node_id:
                    attempt["reward"] = 1.0
    else:
        for node in graph.nodes.values():
            node_grade_prompt = f"""
            Original Problem: {problem_desc}
            Current Step Task: {node.task_description}
            
            [Expected Step Output]: 
            {node.expected_output}
            
            [Model's Actual Step Output]: 
            {node.output_content}
            
            As a strict judge, evaluate whether the model's actual output matches the expected output for this specific step.
            - Output 1 if completely correct.
            - Output 0 if completely incorrect.
            - Output a float between 0.1 and 0.9 for partial correctness.
            Output ONLY the number, with no other characters.
            """
            
            try:
                resp_node_grade = client.chat.completions.create(
                    model=GRADER_MODEL_NAME,
                    messages=[{"role": "user", "content": node_grade_prompt}],
                    max_tokens=10,
                    temperature=0.0,
                )
                node_score_str = resp_node_grade.choices[0].message.content.strip()
                match_node = re.search(r"([0-9]*\.?[0-9]+)", node_score_str)
                node_score = float(match_node.group(1)) if match_node else 0.0
                
                if hasattr(resp_node_grade, 'usage') and resp_node_grade.usage:
                    total_grader_prompt_tokens += resp_node_grade.usage.prompt_tokens
                    total_grader_completion_tokens += resp_node_grade.usage.completion_tokens
                    
            except Exception as e:
                print(f"节点 {node.node_id} 打分失败: {e}")
                node_score = 0.0
            
            node.reward = max(0.0, min(1.0, node_score))
            observations.append((node.feature_vector, node.selected_model, node.reward))
            
            for attempt in execution_attempts:
                if attempt["node_id"] == node.node_id:
                    attempt["reward"] = node.reward

    # 将裁判模型的额外开销附加到第一个尝试记录中，方便后续统一统计
    if execution_attempts:
        execution_attempts[0]["grader_overhead_prompt_tokens"] = total_grader_prompt_tokens
        execution_attempts[0]["grader_overhead_completion_tokens"] = total_grader_completion_tokens

    return observations, final_output, final_score, execution_attempts

In [14]:
class GreedyNeuralUCBModel:
    def __init__(
        self,
        model_names,
        feature_dim=EMBEDDING_DIM,
        learning_rate=LEARNING_RATE, # 对应算法中的步长 (step size) η
        reg=REG,                     # 对应算法中的正则化参数 (Regularization parameter) λ
        m=WIDTH,                     # 对应算法中的网络宽度 (network width) m
        J=GD_STEPS,                  # 对应算法中的梯度下降步数 (gradient descent steps) J
        gamma=GEMMA,                 # 对应算法中的探索系数 (exploration coefficient) γ
        batch_size=BATCH_SIZE,       # 对应算法中的批次大小 (batch size) B_batch
        L=L,                         # 神经网络的层数
        seed=42,
    ):
        self.model_names = model_names
        self.feature_dim = feature_dim
        self.learning_rate = learning_rate
        self.reg = reg
        self.m = m
        self.J = J
        self.gamma = gamma
        self.batch_size = batch_size
        self.L = L
        
        self.model_name_to_index = {name: i for i, name in enumerate(model_names)}
        self.rng = np.random.default_rng(seed)

        # ==========================================
        # 算法 Algorithm 1 第 1 行: Initialize experience buffer B = ∅
        # ==========================================
        self.buffer = [] # 经验回放缓冲区，用于暂存当前的观测数据 (x_vector, model_name, reward)
        self.t = 0       # 记录 query round t，即当前处理了多少个节点

        # 网络结构：输入层 -> (L-1)个隐藏层(宽度为m) -> 输出层(维度1)
        num_hidden = max(self.L - 1, 1)
        layer_sizes = [self.feature_dim] + [self.m] * num_hidden + [1]
        self.layer_sizes = layer_sizes

        def _block_diag(W):
            """辅助函数：构造分块对角矩阵，用于 NTK 初始化"""
            z = np.zeros_like(W)
            top = np.concatenate([W, z], axis=1)
            bottom = np.concatenate([z, W], axis=1)
            return np.concatenate([top, bottom], axis=0)

        def _init_params():
            """
            算法 Algorithm 1 第 1 行: randomly initialize network parameters \theta_{0,j}
            这里采用神经正切核（NTK）的启发式初始化方法，确保初始网络输出接近 0 且梯度行为稳定。
            """
            params = []
            for li, (in_dim, out_dim) in enumerate(zip(layer_sizes[:-1], layer_sizes[1:]), start=1):
                if li < len(layer_sizes) - 1:
                    # 隐藏层的初始化
                    if in_dim == out_dim and in_dim % 2 == 0:
                        half = in_dim // 2
                        W = self.rng.normal(0.0, np.sqrt(4.0 / self.m), size=(half, half)).astype(np.float32)
                        w = _block_diag(W).astype(np.float32)
                    else:
                        w = self.rng.normal(0.0, np.sqrt(4.0 / self.m), size=(out_dim, in_dim)).astype(np.float32)
                    b = np.zeros(out_dim, dtype=np.float32)
                else:
                    # 输出层的初始化
                    if in_dim % 2 == 0:
                        half = in_dim // 2
                        w_vec = self.rng.normal(0.0, np.sqrt(2.0 / self.m), size=(half,)).astype(np.float32)
                        w = np.concatenate([w_vec, -w_vec], axis=0).reshape(1, -1).astype(np.float32)
                        if w.shape[1] != in_dim:
                            w = self.rng.normal(0.0, np.sqrt(2.0 / self.m), size=(out_dim, in_dim)).astype(np.float32)
                    else:
                        w = self.rng.normal(0.0, np.sqrt(2.0 / self.m), size=(out_dim, in_dim)).astype(np.float32)
                    b = np.zeros(out_dim, dtype=np.float32)
                params.append((w, b))
            return params

        def _param_count(params):
            """计算网络参数的总数量，用于初始化 Z 矩阵的大小"""
            return sum(w.size + b.size for w, b in params)

        def _build_net_from_params(params):
            """根据生成的参数构建 PyTorch 神经网络"""
            net = LLMNet(input_dim=self.feature_dim, width=self.m, L=self.L).to(DEVICE)
            linear_layers = [m for m in net.net if isinstance(m, nn.Linear)]
            for (w_np, b_np), layer in zip(params, linear_layers):
                with torch.no_grad():
                    layer.weight.copy_(torch.tensor(w_np, dtype=layer.weight.dtype))
                    layer.bias.copy_(torch.tensor(b_np, dtype=layer.bias.dtype))
            return net

        def _flatten_params(net):
            """将网络的所有参数展平成一个一维向量"""
            return torch.cat([p.detach().reshape(-1) for p in net.parameters()]).cpu().numpy()

        self.models = {}
        # ==========================================
        # 算法 Algorithm 1 第 1 行: For each LLM j in [M], 初始化网络参数和协方差矩阵 Z
        # ==========================================
        for model_name in self.model_names:
            params = _init_params()
            p = _param_count(params)
            net = _build_net_from_params(params)
            theta0 = _flatten_params(net)
            
            self.models[model_name] = {
                "params": params,
                "net": net,
                "theta": np.copy(theta0),  # 当前轮次的网络参数
                "theta0": np.copy(theta0), # 初始网络参数（用于正则化约束，防止灾难性遗忘）
                # 算法要求 set Z_{0,j} = \lambda I。
                # 由于全矩阵太大且求逆太慢，这里使用【对角线近似】(Diagonal Approximation)
                # 即用一个长度为 p 的一维数组代替矩阵，初始值为正则化参数 λ (self.reg)
                "Z_diag": np.ones(p, dtype=np.float32) * self.reg, 
                "last_call_time": 0,
            }

    def add_observations_and_update(self, observations):
        """
        对应算法图第 20-31 行的逻辑：收集真实反馈并更新神经网络参数。
        observations: 一个列表，元素格式为 (特征向量x, 选中的大模型名称, 真实得分reward)
        """
        self.t += 1
        
        # ==========================================
        # 算法 Algorithm 1 第 20 行: Add observation tuples to buffer B
        # ==========================================
        for obs in observations:
            self.buffer.append(obs)
            
        # ==========================================
        # 算法 Algorithm 1 第 21 行: if t mod B_batch == 0 then (是否达到了设定的批次大小)
        # ==========================================
        if self.t % self.batch_size == 0 and len(self.buffer) > 0:
            
            # 算法 Algorithm 1 第 22 行: for each LLM j in [M] do
            for model_name, model_state in self.models.items():
                
                # 算法 Algorithm 1 第 23 行: Let B_j be the subset of buffer B where LLM j was selected
                # 过滤出“当前这个大模型”被选中时产生的数据样本
                B_j = [obs for obs in self.buffer if obs[1] == model_name]
                if not B_j: # 如果这个模型在这个批次里一次都没被选过，就跳过不更新
                    continue
                
                net = model_state["net"]
                net.train() # 开启训练模式
                optimizer = optim.Adam(net.parameters(), lr=self.learning_rate)
                
                # 将该模型专属的经验数据转换为 PyTorch 张量
                xs = torch.tensor(np.array([d[0] for d in B_j]), dtype=torch.float32).to(DEVICE)
                ys = torch.tensor(np.array([d[2] for d in B_j]), dtype=torch.float32).to(DEVICE)
                theta0 = torch.tensor(model_state["theta0"], dtype=torch.float32).to(DEVICE)

                # ==========================================
                # 算法 Algorithm 1 第 25 行: Update \theta_{t,j} 最小化 L2 loss (包含正则项)
                # ==========================================
                # 进行 J 次梯度下降迭代 (for J iterations)
                for _ in range(self.J):
                    optimizer.zero_grad()
                    preds = net(xs)
                    
                    # MSE 损失: 1/2 * (预测值 - 真实得分)^2
                    mse = 0.5 * torch.mean((preds - ys) ** 2)
                    
                    # NTK 理论要求的正则项: m * λ / 2 * ||\theta - \theta_0||^2
                    # 这个项可以防止网络参数偏离初始值太远，保证理论收敛性
                    theta = torch.cat([p.reshape(-1) for p in net.parameters()])
                    reg_term = 0.5 * self.m * self.reg * torch.sum((theta - theta0) ** 2)
                    
                    # 总损失 = 预测误差 + 正则项
                    loss = mse + reg_term
                    loss.backward()
                    optimizer.step()

                # 保存更新后的网络参数
                model_state["theta"] = torch.cat([p.detach().reshape(-1) for p in net.parameters()]).cpu().numpy()

                # ==========================================
                # 算法 Algorithm 1 第 24 行: Update Z_{t,j} 协方差矩阵（用于后续计算探索奖励）
                # ==========================================
                for x_val in xs:
                    x_single = x_val.unsqueeze(0)
                    net.zero_grad(set_to_none=True)
                    # 前向传播求值
                    y = net(x_single).sum()
                    
                    # 反向传播求【网络输出对输入参数的梯度】 g = \nabla f(x; \theta)
                    grads = torch.autograd.grad(y, net.parameters(), retain_graph=False, create_graph=False)
                    g = torch.cat([grad.reshape(-1) for grad in grads]).detach().cpu().numpy()
                    
                    # 理论上是 Z = Z + g * g^T / m。由于我们用了对角线近似，这里变成了向量逐元素的平方累加
                    model_state["Z_diag"] += (g ** 2) / float(self.m)
            
            # ==========================================
            # 算法 Algorithm 1 第 27 行: Clear buffer B = ∅
            # ==========================================
            self.buffer = []

    def select_model(self, feature_vector):
        """
        对应算法图第 7-11 行的逻辑：遍历所有大模型，计算 UCB(上限置信区间) 分数，选出得分最高的模型。
        """
        best_model = None
        max_ucb = -float('inf')
        model_scores_info = {}

        # 构造节点特征向量 x_{t,n}
        x = torch.tensor(feature_vector, dtype=torch.float32).to(DEVICE)
        if x.dim() == 1:
            x = x.unsqueeze(0)

        # 算法 Algorithm 1 第 7 行: for each LLM j in [M] do
        for model_name, model_state in self.models.items():
            net = model_state["net"]
            Z_diag = model_state["Z_diag"] 

            net.eval() # 开启评估模式，关闭 Dropout 等
            net.zero_grad(set_to_none=True) 
            
            # ==========================================
            # 算法 Algorithm 1 第 8 行: Compute estimated score \hat{v}_{t,n,j}
            # ==========================================
            # 预估得分 = 神经网络的输出分数
            y_pred_tensor = net(x).sum()
            v_hat = y_pred_tensor.item() 

            # 求预测值对网络参数的梯度 g = \nabla f(x; \theta)
            grads = torch.autograd.grad(y_pred_tensor, net.parameters(), retain_graph=False, create_graph=False)
            g = torch.cat([grad.reshape(-1) for grad in grads]).detach().cpu().numpy()

            # ==========================================
            # 算法 Algorithm 1 第 9 行: Compute UCB u_{t,n,j}
            # ==========================================
            # UCB = 预估得分 + 探索奖励(Bonus)
            # 原始公式里的矩阵运算 Z^{-1} 被简化为了向量点除 (Z_diag + 1e-8 防止除零)
            bonus = self.gamma * np.sqrt(np.sum((g ** 2) / (self.m * (Z_diag + 1e-8))))
            ucb_score = v_hat + bonus
            
            model_scores_info[model_name] = {
                "pred_score": round(v_hat, 4),
                "bonus": round(bonus, 4),
                "total_ucb": round(ucb_score, 4)
            }

            # ==========================================
            # 算法 Algorithm 1 第 11 行: Select LLM with max UCB
            # ==========================================
            if ucb_score > max_ucb:
                max_ucb = ucb_score
                best_model = model_name

        # 返回选出的最强模型名称，以及所有模型的打分细节
        return best_model, model_scores_info

In [ ]:
def main_training_loop(dataset: list, ucb_model, feature_extractor, client, recorder):
    """
    运行基线/消融实验主循环：使用 Ground Truth DAG 结构
    """
    for idx, record in enumerate(tqdm(dataset, desc="Processing Queries")):
        problem_id = record.get("problem_id", str(idx))
        problem_desc = record.get("problem_description", "")
        
        if not problem_desc:
            continue
            
        # 1. 静态构建：直接从 JSON 数据构建 Ground Truth DAG
        graph = build_dag_from_ground_truth(record)
        
        # 2. 图执行与评估（内部已经根据你的要求处理了单步 reward 的分配）
        obs, final_out, score, attempts = execute_and_evaluate_dag(
            graph=graph,
            ucb_model=ucb_model,
            feature_extractor=feature_extractor,
            client=client
        )
        
        # 提取并记录 JSON 中所有的 expected_output，方便在 record 文件中人工核对
        expected_answers = "\n".join([n.get("expected_output", "") for n in record.get("nodes", [])])
            
        recorder.add_record(
            problem_id=str(problem_id),
            question=problem_desc,
            expected_answers=expected_answers,
            final_status=str(score),
            attempts=attempts
        )
        
        # 3. 更新 UCB 模型参数
        ucb_model.add_observations_and_update(obs)
        
    print("🎉 所有查询处理并训练完毕！")

# ==========================================
# 🚀 启动执行模块 (可自由配置测试数据量)
# ==========================================
# 1. 实例化核心组件
ucb_model = GreedyNeuralUCBModel(model_names=AVAILABLE_LLMS)
# 为了与之前的全量测试区分，换一个记录文件名
recorder = ResultRecorder("execution_records_ablation.json")

# 2. 读取 DAG 形式的数据集 (请确保路径和你的项目结构匹配)
my_dag_dataset = load_dataset("dag_math_dag.json") 

# ==========================================
# 💡 在这里自由设置你想测试的数据集大小！
# ==========================================
TEST_SIZE = 2  # <--- 修改这里的数字控制测试量。如果想跑全量，改为 TEST_SIZE = None

if TEST_SIZE is not None:
    dataset_to_run = my_dag_dataset[:TEST_SIZE]
else:
    dataset_to_run = my_dag_dataset

print(f"准备实验，当前设定测试数据量: {len(dataset_to_run)} 条")

# 3. 开始端到端运行
main_training_loop(dataset_to_run, ucb_model, extractor, client, recorder)

✅ 成功加载数据集: dag_math_dag.json
准备实验，当前设定测试数据量: 2 条


Processing Queries:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Successfully loaded Ground Truth DAG with 8 nodes for Problem 0.


c:\Users\GrayB\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\autograd\graph.py:825: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\cuda\CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Processing Queries:  50%|█████     | 1/2 [05:58<05:58, 358.80s/it]

✅ Successfully loaded Ground Truth DAG with 9 nodes for Problem 1.
